In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

In [3]:
df = pd.read_csv(
    "../dataset/APACHE 1.xlsx - Sheet2.csv"
)

print(df.shape)

df.head()

(104557, 67)


,ID,UHID,IPNumber,ICUChartDate,Age,Temperature,MeanArterialPressure,HeartRate,RespiratoryRate,FiO2,...,AdmissionDate,UNIT_ID,CREATEDON,RNK,APACHE_WARD,CUSTOMERSTATUS,LOCATIONID,LOCATION,DISCHARGEDATE,PERIOD_WID
0,4978,APD1.0012042887,DELIP573294,1/23/26 0:00,83,36.9,103.00,100.0,20.0,40.0,...,1/21/26 0:00,4,2026-02-10T01:36:11.1029417Z,1,6th Flr T2,ALIVE,10701.0,Delhi - Sarita Vihar,10:00.4,20260123
1,10124,AHS.0000417587,AHSIP101940,2/4/26 0:00,44,36.0,84.00,78.0,20.0,30.0,...,2/3/26 0:00,8,2026-02-10T01:38:57.3490126Z,1,GENERAL WARD LEVEL 5,ALIVE,10391.0,Apollo hospital Sheshadripuram,NaN,20260204
2,36594,AHLG.000582499,AHLGIP68174,2/3/26 0:00,75,36.7,73.26,104.0,20.0,30.0,...,2/2/26 0:00,1,2026-02-10T01:33:13.9669818Z,1,NEURO ICU,ALIVE,17001.0,Assam Hospitals Limited – Guwahati,NaN,20260203
3,4815,ANM1.0001123066,ANMIP185974,1/19/26 0:00,43,37.0,77.00,110.0,16.0,40.0,...,1/19/26 0:00,5,2026-02-10T01:38:05.9903312Z,1,D WARD,DEAD,10551.0,Navi Mumbai,NaN,20260119
4,10027,AHS.0000416624,AHSIP101703,1/25/26 0:00,51,36.0,98.00,96.0,20.0,30.0,...,1/25/26 0:00,8,2026-02-10T01:38:57.3490126Z,1,EMERGENCY,ALIVE,10391.0,Apollo hospital Sheshadripuram,01:18.2,20260125


In [4]:
features = [

    'Age',
    'Temperature',
    'MeanArterialPressure',
    'HeartRate',
    'RespiratoryRate',
    'FiO2',
    'pO2',
    'pCO2',
    'ArterialpH',
    'Sodium',
    'Creatinine',
    'Urea',
    'Albumin',
    'Bilirubin',
    'Hematocrit',
    'WBC',
    'ApacheivScore',
    'ApsScore'
]

In [5]:
los_df = df[
    features + ['EstimatedLengthOfStay']
].copy()

In [6]:
los_df.dropna(

    subset=['EstimatedLengthOfStay'],

    inplace=True
)

In [7]:
def los_category(days):

    if days <= 3:

        return 0

    elif days <= 7:

        return 1

    else:

        return 2

In [8]:
los_df['LOS_Class'] = los_df[
    'EstimatedLengthOfStay'
].apply(los_category)

In [9]:
imputer = SimpleImputer(
    strategy='median'
)

los_df[features] = imputer.fit_transform(
    los_df[features]
)

In [10]:
X = los_df[features]

y = los_df['LOS_Class']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42
)

In [12]:
model = XGBClassifier(

    n_estimators=300,

    learning_rate=0.05,

    max_depth=5,

    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("LOS CLASSIFIER TRAINED")

LOS CLASSIFIER TRAINED


In [13]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("LOS ACCURACY:", accuracy)

print(
    classification_report(
        y_test,
        y_pred
    )
)

LOS ACCURACY: 0.8359793420045907
              precision    recall  f1-score   support

           0       0.86      0.89      0.87      9589
           1       0.82      0.85      0.84     10337
           2       0.62      0.15      0.25       986

    accuracy                           0.84     20912
   macro avg       0.76      0.63      0.65     20912
weighted avg       0.83      0.84      0.83     20912



In [14]:
joblib.dump(

    model,

    "../models/los_classifier.pkl"
)

joblib.dump(

    features,

    "../models/features.pkl"
)

joblib.dump(

    imputer,

    "../models/imputer.pkl"
)

print("LOS CLASSIFIER SAVED")

LOS CLASSIFIER SAVED
